# GPU Final Notebook - Math Carry

Modernized final arithmetic run using behavior-specific math SAEs plus the current contrast-target, all-position swap, layer-scan, and graph-positive sparse intervention methods.

This notebook trains or resumes final addition-specific SAEs in `sae_checkpoints_math_final_train`, then saves distinct `math_final_*` graph/intervention outputs.


## Step 1 - Mount Drive and fetch code


In [7]:
# Step 1: Mount Google Drive and fetch project code
# GitHub is the primary source for code. Drive project.zip is kept as a backup.
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import subprocess

repo_url = "https://github.com/evey-dev/test_run.git"
repo_dir = "/content/test_run"
use_github_code = True

def run_cmd(cmd):
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True)

github_ok = False
if use_github_code:
    try:
        clone_dir = repo_dir
        if os.path.isdir(os.path.join(clone_dir, ".git")):
            run_cmd(["git", "-C", clone_dir, "pull", "--ff-only"])
        else:
            if os.path.exists(clone_dir) and os.listdir(clone_dir):
                clone_dir = "/content/test_run_github"
            if os.path.isdir(os.path.join(clone_dir, ".git")):
                run_cmd(["git", "-C", clone_dir, "pull", "--ff-only"])
            elif os.path.exists(clone_dir) and os.listdir(clone_dir):
                raise RuntimeError(f"{clone_dir} exists but is not a git repo")
            else:
                run_cmd(["git", "clone", "--depth", "1", repo_url, clone_dir])
        os.chdir(clone_dir)
        github_ok = True
        print(f"Using GitHub checkout: {os.getcwd()}")
    except Exception as exc:
        print("GitHub checkout failed; falling back to Drive project.zip.")
        print(repr(exc))

if not github_ok:
    zip_path = "/content/drive/MyDrive/mphil-project/project.zip"
    if not os.path.exists(zip_path):
        raise FileNotFoundError(f"Could not find {zip_path}. Fix GitHub access or upload project.zip.")
    print(f"Extracting backup zip {zip_path}...")
    run_cmd(["unzip", "-q", "-o", zip_path, "-d", "/content/"])
    for candidate in ["/content/test_run", "/content/mphil_project/test_run", "/content"]:
        if os.path.isdir(os.path.join(candidate, "src")) and os.path.isdir(os.path.join(candidate, "configs")):
            os.chdir(candidate)
            break
    else:
        raise FileNotFoundError("Could not find extracted project root containing src/ and configs/.")
    print(f"Using Drive backup project: {os.getcwd()}")

print(f"Current working directory: {os.getcwd()}")
!ls -l


Mounted at /content/drive
$ git -C /content/test_run pull --ff-only
Using GitHub checkout: /content/test_run
Current working directory: /content/test_run
total 780
-rw-r--r-- 1 root root   6680 Jul 10 04:34 ai_handoff_summary.txt
-rw-r--r-- 1 root root   8894 Jul 10 04:34 changes_summary.txt
drwxr-xr-x 2 root root   4096 Jul 10 04:34 configs
drwxr-xr-x 2 root root   4096 Jul 10 04:34 data
-rw-r--r-- 1 root root    264 Jul 10 04:34 environment.yml
-rw-r--r-- 1 root root  19186 Jul 10 04:34 IMPLEMENTATION_PLAN.md
-rw-r--r-- 1 root root   1579 Jul 10 04:34 Instructions.md
drwxr-xr-x 4 root root   4096 Jul 10 04:36 mechanistic_data
drwxr-xr-x 2 root root   4096 Jul 10 04:36 mechanistic_data_math_final_train
drwxr-xr-x 2 root root   4096 Jul 10 04:34 mechanistic_interpretability_repro.egg-info
drwxr-xr-x 3 root root   4096 Jul 10 06:00 outputs
-rw-r--r-- 1 root root   6551 Jul 10 04:34 project.txt
-rw-r--r-- 1 root root  11435 Jul 10 04:34 README.md
-rw-r--r-- 1 root root  12564 Jul 10 04:3

## Step 2 - Install dependencies


In [8]:
# Step 2: Install dependencies
!pip install --upgrade "transformers>=4.51.0" accelerate
!pip install -e .


Obtaining file:///content/test_run
  Preparing metadata (setup.py) ... done
  Attempting uninstall: mechanistic-interpretability-repro
    Found existing installation: mechanistic-interpretability-repro 0.1.0
    Uninstalling mechanistic-interpretability-repro-0.1.0:
      Successfully uninstalled mechanistic-interpretability-repro-0.1.0
  Running setup.py develop for mechanistic-interpretability-repro


## Step 3 - Generate addition dataset


In [3]:
# Step 3: Generate addition dataset
!python data/generate_datasets.py
print("Dataset files:")
!ls -lh data/addition_data.csv


Generating Datasets...

Skipping capital sentence generation. (Pass --capitals flag if needed).
Standalone Mode complete:
Saved 1000 addition problems and 
Saved 1000 unit problems.
Dataset files:
-rw-r--r-- 1 root root 52K Jul 10 04:34 data/addition_data.csv


## Step 4 - Restore final math SAE checkpoints from Drive if available


In [ ]:
# Restore final math-specific SAE checkpoints from Drive if they already exist.
# If this succeeds, the capture/training cells below will skip themselves.
from pathlib import Path
import shutil

FINAL_LAYERS = [4, 8, 12, 16, 20, 24, 28]
local_sae_dir = Path('/content/test_run/mechanistic_data/sae_checkpoints_math_final_train')
drive_sae_dir = Path('/content/drive/MyDrive/mphil-project/mechanistic_data/sae_checkpoints_math_final_train')

def has_complete_sae_set(path, layers):
    path = Path(path)
    return all(
        (path / f'sae_layer{layer}.pt').exists()
        and (path / f'sae_layer{layer}_metadata.json').exists()
        and (path / f'latents_layer{layer}.npy').exists()
        for layer in layers
    )

MATH_FINAL_SAES_READY = False
if has_complete_sae_set(drive_sae_dir, FINAL_LAYERS):
    local_sae_dir.mkdir(parents=True, exist_ok=True)
    for src in sorted(drive_sae_dir.iterdir()):
        if src.is_file():
            shutil.copy2(src, local_sae_dir / src.name)
    MATH_FINAL_SAES_READY = has_complete_sae_set(local_sae_dir, FINAL_LAYERS)
    print(f'Restored complete final math SAE checkpoint set from Drive: {drive_sae_dir}')
    print(f'Local checkpoint dir: {local_sae_dir}')
else:
    print(f'No complete final math SAE checkpoint set found at {drive_sae_dir}.')
    print('The notebook will capture activations and train SAEs below.')

print('MATH_FINAL_SAES_READY =', MATH_FINAL_SAES_READY)


## Step 5 - Capture math-only activations


In [ ]:
# Capture math-only MLP activations for final SAE training, unless checkpoints were restored.
import subprocess

if globals().get('MATH_FINAL_SAES_READY', False):
    print('Skipping activation capture because final math SAE checkpoints were restored from Drive.')
else:
    subprocess.run([
        'python', 'src/capture_activations.py',
        '--output-dir', 'mechanistic_data_math_final_train',
        '--behaviours', 'addition',
        '--layers', '4', '8', '12', '16', '20', '24', '28',
        '--seed', '787',
    ], check=True)
    print('Final math activation bundle:')
    subprocess.run(['ls', '-lh', 'mechanistic_data_math_final_train'], check=True)


## Step 6 - Train final math-specific SAEs


In [ ]:
# Train final math-specific SAEs for all selected layers, unless checkpoints were restored.
import subprocess

if globals().get('MATH_FINAL_SAES_READY', False):
    print('Skipping SAE training because final math SAE checkpoints were restored from Drive.')
else:
    subprocess.run([
        'python', 'src/train.py',
        '--config', 'configs/sae_math_final_train_config.yaml',
        '--drive-dir', '/content/drive/MyDrive/mphil-project/mechanistic_data/sae_checkpoints_math_final_train',
    ], check=True)

print('Final math local checkpoints:')
subprocess.run(['ls', '-lh', 'mechanistic_data/sae_checkpoints_math_final_train'], check=True)
print('Final math Drive checkpoints:')
subprocess.run(['ls', '-lh', '/content/drive/MyDrive/mphil-project/mechanistic_data/sae_checkpoints_math_final_train'], check=True)


## Step 7 - Sanity check the carry/no-carry minimal pair


In [6]:
# Confirm token positions for the two prompts before all-position swaps.
# The prompts should differ only at the first operand's ones digit: 54 vs 58.
!python src/intervention.py \
  --mode inhibit \
  --prompt "Question: What is 58 + 83? Answer: 1" \
  --print-tokens

!python src/intervention.py \
  --mode inhibit \
  --prompt "Question: What is 54 + 83? Answer: 1" \
  --print-tokens


Loading model...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 186.69it/s]
Prompt token positions:
   0: id=14582    token='Question'
   1: id=25       token=':'
   2: id=3555     token=' What'
   3: id=374      token=' is'
   4: id=220      token=' '
   5: id=20       token='5'
   6: id=23       token='8'
   7: id=488      token=' +'
   8: id=220      token=' '
   9: id=23       token='8'
  10: id=18       token='3'
  11: id=30       token='?'
  12: id=21806    token=' Answer'
  13: id=25       token=':'
  14: id=220      token=' '
  15: id=16       token='1'
Use --positions last, --positions all, or comma-separated indices such as --positions 4,5,9
Loading model...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 186.81it/s]
Prompt token positions:
   0: id

## Step 8 - Rebuild contrast attribution graphs


In [7]:
# Carry graph: 58 + 83 = 141, tens digit should be 4 rather than 3.
!python src/attribution_graph.py \
  --prompt "Question: What is 58 + 83? Answer: 1" \
  --target "4" \
  --contrast-target "3" \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_math_final_train_config.yaml \
  --top-k-nodes 20 --top-k-edges 30 \
  --output-json outputs/math_final_carry_58_83_4v3_graph.json \
  --output-html outputs/math_final_carry_58_83_4v3_graph.html \
  --output-mermaid outputs/math_final_carry_58_83_4v3_graph.md


Loading model and tokenizer...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 186.24it/s]
Disabling gradients for all model parameters for speed and memory efficiency.
Loading SAE models from /content/test_run/mechanistic_data/sae_checkpoints_math_final_train to device: cuda...
Loaded SAE for layer 4 (scaling factor: 0.1420)
Loaded SAE for layer 8 (scaling factor: 0.2652)
Loaded SAE for layer 12 (scaling factor: 0.2661)
Loaded SAE for layer 16 (scaling factor: 0.3459)
Loaded SAE for layer 20 (scaling factor: 0.3685)
Loaded SAE for layer 24 (scaling factor: 0.7766)
Loaded SAE for layer 28 (scaling factor: 0.8645)
Running forward pass...

Top 5 model predictions:
  '4' (prob: 1.0000, logit: 30.62)
  '3' (prob: 0.0008, logit: 23.50)
  '2' (prob: 0.0003, logit: 22.38)
  '1' (prob: 0.0001, logit: 21.75)
  '0' (prob: 0.0001, logit: 21.50)

Target token for attribution: '4' (ID:

In [8]:
# No-carry graph: 54 + 83 = 137, tens digit should be 3 rather than 4.
!python src/attribution_graph.py \
  --prompt "Question: What is 54 + 83? Answer: 1" \
  --target "3" \
  --contrast-target "4" \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_math_final_train_config.yaml \
  --top-k-nodes 20 --top-k-edges 30 \
  --output-json outputs/math_final_nocarry_54_83_3v4_graph.json \
  --output-html outputs/math_final_nocarry_54_83_3v4_graph.html \
  --output-mermaid outputs/math_final_nocarry_54_83_3v4_graph.md


Loading model and tokenizer...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 179.66it/s]
Disabling gradients for all model parameters for speed and memory efficiency.
Loading SAE models from /content/test_run/mechanistic_data/sae_checkpoints_math_final_train to device: cuda...
Loaded SAE for layer 4 (scaling factor: 0.1420)
Loaded SAE for layer 8 (scaling factor: 0.2652)
Loaded SAE for layer 12 (scaling factor: 0.2661)
Loaded SAE for layer 16 (scaling factor: 0.3459)
Loaded SAE for layer 20 (scaling factor: 0.3685)
Loaded SAE for layer 24 (scaling factor: 0.7766)
Loaded SAE for layer 28 (scaling factor: 0.8645)
Running forward pass...

Top 5 model predictions:
  '3' (prob: 1.0000, logit: 29.38)
  '2' (prob: 0.0003, logit: 21.25)
  '4' (prob: 0.0002, logit: 20.88)
  '1' (prob: 0.0001, logit: 20.00)
  '0' (prob: 0.0001, logit: 19.88)

Target token for attribution: '3' (ID:

In [9]:
# Persist final math graphs immediately, because graph construction is the expensive part.
import os, shutil, glob

drive_out = '/content/drive/MyDrive/mphil-project/outputs'
os.makedirs(drive_out, exist_ok=True)
for src in glob.glob('/content/test_run/outputs/math_final_*_graph.*'):
    if os.path.isfile(src):
        shutil.copy2(src, drive_out)
        print('Copied', os.path.basename(src))
print('Done!')


Copied math_final_nocarry_54_83_3v4_graph.json
Copied math_final_carry_58_83_4v3_graph.html
Copied math_final_nocarry_54_83_3v4_graph.md
Copied math_final_carry_58_83_4v3_graph.md
Copied math_final_nocarry_54_83_3v4_graph.html
Copied math_final_carry_58_83_4v3_graph.json
Done!


## Step 9 - Component-level controls


In [10]:
# All-position MLP knockout on the carry prompt.
# This is the strongest math diagnostic from the previous notebook.
!python src/intervention.py \
  --mode inhibit \
  --prompt "Question: What is 58 + 83? Answer: 1" \
  --target-token "4, 3, 2" \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_math_final_train_config.yaml \
  --full-knockout \
  --knockout-component mlp \
  --positions all \
  --layer-scan \
  --output outputs/math_final_knockout_mlp_allpos_carry_58_83.json


Loading model...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 186.07it/s]

[1/3] Clean Model Baseline:
  - Top prediction: '4' (prob: 1.0000, logit: 30.62)
  - Target ' ': prob: 0.0000, logit: 14.31
  - Target '2': prob: 0.0003, logit: 22.38
  - Target '3': prob: 0.0008, logit: 23.50
  - Target '4': prob: 1.0000, logit: 30.62

[2/3] Running SAE Reconstruction-only Baseline (no features inhibited)...
Running forward pass with inhibition at positions [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]: {}...
Editing token positions:
   0: id=14582    token='Question'
   1: id=25       token=':'
   2: id=3555     token=' What'
   3: id=374      token=' is'
   4: id=220      token=' '
   5: id=20       token='5'
   6: id=23       token='8'
   7: id=488      token=' +'
   8: id=220      token=' '
   9: id=23       token='8'
  10: id=18       token='3'
  11: id=30       to

In [11]:
# Progressive sparse ablation of graph-positive carry features.
!python src/intervention.py \
  --mode inhibit \
  --prompt "Question: What is 58 + 83? Answer: 1" \
  --target-token "4, 3, 2" \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_math_final_train_config.yaml \
  --graph-json outputs/math_final_carry_58_83_4v3_graph.json \
  --graph-feature-sign positive \
  --scan \
  --positions all \
  --output outputs/math_final_scan_carry_positive_allpos.json


Loading model...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 187.35it/s]
Loaded 157 nodes from graph JSON; extracted features across 7 layers (73 total features)
  Graph feature sign filter: positive (skipped 67 feature nodes)

[1/3] Clean Model Baseline:
  - Top prediction: '4' (prob: 1.0000, logit: 30.62)
  - Target ' ': prob: 0.0000, logit: 14.31
  - Target '2': prob: 0.0003, logit: 22.38
  - Target '3': prob: 0.0008, logit: 23.50
  - Target '4': prob: 1.0000, logit: 30.62

[2/3] Running SAE Reconstruction-only Baseline (no features inhibited)...
Running forward pass with inhibition at positions [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]: {}...
Editing token positions:
   0: id=14582    token='Question'
   1: id=25       token=':'
   2: id=3555     token=' What'
   3: id=374      token=' is'
   4: id=220      token=' '
   5: id=20       token='5'
   6: i

## Step 10 - Full latent minimal-pair swaps


In [12]:
# Patch NO-CARRY (54+83 -> tens 3) into CARRY (58+83 -> tens 4).
# Desired causal direction: 4 falls and 3 rises.
!python src/intervention.py \
  --mode swap \
  --source-prompt "Question: What is 54 + 83? Answer: 1" \
  --prompt "Question: What is 58 + 83? Answer: 1" \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_math_final_train_config.yaml \
  --target-token "4, 3, 2" \
  --positions all \
  --layer-scan \
  --output outputs/math_final_swap_full_nocarry_to_carry.json


Loading model...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 186.03it/s]

[1/3] Clean Model Baseline:
  - Top prediction: '4' (prob: 1.0000, logit: 30.62)
  - Target ' ': prob: 0.0000, logit: 14.31
  - Target '2': prob: 0.0003, logit: 22.38
  - Target '3': prob: 0.0008, logit: 23.50
  - Target '4': prob: 1.0000, logit: 30.62

Source baseline prediction on 'Question: What is 54 + 83? Answer: 1':
  - Target ' ': prob: 0.0000, logit: 14.44
  - Target '2': prob: 0.0003, logit: 21.25
  - Target '3': prob: 1.0000, logit: 29.38
  - Target '4': prob: 0.0002, logit: 20.88

Running Swap-In Intervention (swapping FULL latent code at every requested layer from source to target at positions 'all'; edit_strength=1.0)...
Capturing source activations on prompt: 'Question: What is 54 + 83? Answer: 1' at positions [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]...
Running target 

In [13]:
# Reverse direction: patch CARRY (58+83 -> tens 4) into NO-CARRY (54+83 -> tens 3).
# Desired causal direction: 3 falls and 4 rises.
!python src/intervention.py \
  --mode swap \
  --source-prompt "Question: What is 58 + 83? Answer: 1" \
  --prompt "Question: What is 54 + 83? Answer: 1" \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_math_final_train_config.yaml \
  --target-token "4, 3, 2" \
  --positions all \
  --layer-scan \
  --output outputs/math_final_swap_full_carry_to_nocarry.json


Loading model...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 185.50it/s]

[1/3] Clean Model Baseline:
  - Top prediction: '3' (prob: 1.0000, logit: 29.38)
  - Target ' ': prob: 0.0000, logit: 14.44
  - Target '2': prob: 0.0003, logit: 21.25
  - Target '3': prob: 1.0000, logit: 29.38
  - Target '4': prob: 0.0002, logit: 20.88

Source baseline prediction on 'Question: What is 58 + 83? Answer: 1':
  - Target ' ': prob: 0.0000, logit: 14.31
  - Target '2': prob: 0.0003, logit: 22.38
  - Target '3': prob: 0.0008, logit: 23.50
  - Target '4': prob: 1.0000, logit: 30.62

Running Swap-In Intervention (swapping FULL latent code at every requested layer from source to target at positions 'all'; edit_strength=1.0)...
Capturing source activations on prompt: 'Question: What is 58 + 83? Answer: 1' at positions [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]...
Running target 

## Step 11 - Sparse graph-feature swaps


In [14]:
# Sparse version of NO-CARRY -> CARRY, using no-carry graph-positive features.
!python src/intervention.py \
  --mode swap \
  --source-prompt "Question: What is 54 + 83? Answer: 1" \
  --prompt "Question: What is 58 + 83? Answer: 1" \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_math_final_train_config.yaml \
  --graph-json outputs/math_final_nocarry_54_83_3v4_graph.json \
  --graph-feature-sign positive \
  --target-token "4, 3, 2" \
  --positions all \
  --output outputs/math_final_swap_sparse_nocarry_to_carry.json


Loading model...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 186.09it/s]
Loaded 157 nodes from graph JSON; extracted features across 7 layers (80 total features)
  Graph feature sign filter: positive (skipped 60 feature nodes)

[1/3] Clean Model Baseline:
  - Top prediction: '4' (prob: 1.0000, logit: 30.62)
  - Target ' ': prob: 0.0000, logit: 14.31
  - Target '2': prob: 0.0003, logit: 22.38
  - Target '3': prob: 0.0008, logit: 23.50
  - Target '4': prob: 1.0000, logit: 30.62

Source baseline prediction on 'Question: What is 54 + 83? Answer: 1':
  - Target ' ': prob: 0.0000, logit: 14.44
  - Target '2': prob: 0.0003, logit: 21.25
  - Target '3': prob: 1.0000, logit: 29.38
  - Target '4': prob: 0.0002, logit: 20.88

Running Swap-In Intervention (swapping features {4: [5701, 6519, 6202, 7626, 1649, 6696, 7067], 8: [7596, 4538, 1068, 2120, 2092, 7490, 7162, 6935, 6311, 67

In [15]:
# Sparse reverse: CARRY -> NO-CARRY, using carry graph-positive features.
!python src/intervention.py \
  --mode swap \
  --source-prompt "Question: What is 58 + 83? Answer: 1" \
  --prompt "Question: What is 54 + 83? Answer: 1" \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_math_final_train_config.yaml \
  --graph-json outputs/math_final_carry_58_83_4v3_graph.json \
  --graph-feature-sign positive \
  --target-token "4, 3, 2" \
  --positions all \
  --output outputs/math_final_swap_sparse_carry_to_nocarry.json


Loading model...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 184.98it/s]
Loaded 157 nodes from graph JSON; extracted features across 7 layers (73 total features)
  Graph feature sign filter: positive (skipped 67 feature nodes)

[1/3] Clean Model Baseline:
  - Top prediction: '3' (prob: 1.0000, logit: 29.38)
  - Target ' ': prob: 0.0000, logit: 14.44
  - Target '2': prob: 0.0003, logit: 21.25
  - Target '3': prob: 1.0000, logit: 29.38
  - Target '4': prob: 0.0002, logit: 20.88

Source baseline prediction on 'Question: What is 58 + 83? Answer: 1':
  - Target ' ': prob: 0.0000, logit: 14.31
  - Target '2': prob: 0.0003, logit: 22.38
  - Target '3': prob: 0.0008, logit: 23.50
  - Target '4': prob: 1.0000, logit: 30.62

Running Swap-In Intervention (swapping features {4: [2813, 9, 6519, 6696, 4835, 7292, 1649, 142, 2597, 2377, 1014, 4485, 7067, 1740], 8: [2092, 3456, 677, 7

## Step 12 - Final swap-strength diagnostics

These cells test whether the weak full-latent swap is because the SAE edit is too small, or because SAE latent reconstruction misses the decisive carry direction. They do not require rebuilding attribution graphs.


In [9]:
# Stronger SAE latent swap: NO-CARRY -> CARRY with edit_strength=2.
# If the 3 logit rises much more, the original SAE swap may have been underpowered.
# If it still barely moves, the carry direction is probably not captured well by this SAE latent edit.
!python src/intervention.py \
  --mode swap \
  --source-prompt "Question: What is 54 + 83? Answer: 1" \
  --prompt "Question: What is 58 + 83? Answer: 1" \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_math_final_train_config.yaml \
  --target-token "4, 3, 2" \
  --positions all \
  --edit-strength 2.0 \
  --layer-scan \
  --output outputs/math_final_swap_full_nocarry_to_carry_strength2.json


Loading model...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 186.02it/s]

[1/3] Clean Model Baseline:
  - Top prediction: '4' (prob: 1.0000, logit: 30.62)
  - Target ' ': prob: 0.0000, logit: 14.31
  - Target '2': prob: 0.0003, logit: 22.38
  - Target '3': prob: 0.0008, logit: 23.50
  - Target '4': prob: 1.0000, logit: 30.62

Source baseline prediction on 'Question: What is 54 + 83? Answer: 1':
  - Target ' ': prob: 0.0000, logit: 14.44
  - Target '2': prob: 0.0003, logit: 21.25
  - Target '3': prob: 1.0000, logit: 29.38
  - Target '4': prob: 0.0002, logit: 20.88

Running Swap-In Intervention (swapping FULL latent code at every requested layer from source to target at positions 'all'; edit_strength=2.0)...
Capturing source activations on prompt: 'Question: What is 54 + 83? Answer: 1' at positions [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]...
Running target 

In [10]:
# Raw MLP swap: NO-CARRY -> CARRY.
# This bypasses the SAE latent reconstruction and patches the actual MLP outputs.
# If this works much better than the SAE swap, the SAE decomposition is the bottleneck.
!python src/intervention.py \
  --mode swap \
  --source-prompt "Question: What is 54 + 83? Answer: 1" \
  --prompt "Question: What is 58 + 83? Answer: 1" \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_math_final_train_config.yaml \
  --target-token "4, 3, 2" \
  --positions all \
  --raw-mlp-swap \
  --layer-scan \
  --output outputs/math_final_swap_raw_mlp_nocarry_to_carry.json


Loading model...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 186.06it/s]

[1/3] Clean Model Baseline:
  - Top prediction: '4' (prob: 1.0000, logit: 30.62)
  - Target ' ': prob: 0.0000, logit: 14.31
  - Target '2': prob: 0.0003, logit: 22.38
  - Target '3': prob: 0.0008, logit: 23.50
  - Target '4': prob: 1.0000, logit: 30.62

Source baseline prediction on 'Question: What is 54 + 83? Answer: 1':
  - Target ' ': prob: 0.0000, logit: 14.44
  - Target '2': prob: 0.0003, logit: 21.25
  - Target '3': prob: 1.0000, logit: 29.38
  - Target '4': prob: 0.0002, logit: 20.88

Running Swap-In Intervention (swapping RAW MLP output at every requested layer from source to target at positions 'all'; edit_strength=1.0)...
Capturing source activations on prompt: 'Question: What is 54 + 83? Answer: 1' at positions [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]...
Running target pa

In [12]:
# Raw MLP reverse check: CARRY -> NO-CARRY.
# Desired direction: 4 rises relative to 3 on the no-carry prompt.
!python src/intervention.py \
  --mode swap \
  --source-prompt "Question: What is 58 + 83? Answer: 1" \
  --prompt "Question: What is 54 + 83? Answer: 1" \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_math_final_train_config.yaml \
  --target-token "4, 3, 2" \
  --positions all \
  --raw-mlp-swap \
  --layer-scan \
  --output outputs/math_final_swap_raw_mlp_carry_to_nocarry.json


Loading model...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 185.41it/s]

[1/3] Clean Model Baseline:
  - Top prediction: '3' (prob: 1.0000, logit: 29.38)
  - Target ' ': prob: 0.0000, logit: 14.44
  - Target '2': prob: 0.0003, logit: 21.25
  - Target '3': prob: 1.0000, logit: 29.38
  - Target '4': prob: 0.0002, logit: 20.88

Source baseline prediction on 'Question: What is 58 + 83? Answer: 1':
  - Target ' ': prob: 0.0000, logit: 14.31
  - Target '2': prob: 0.0003, logit: 22.38
  - Target '3': prob: 0.0008, logit: 23.50
  - Target '4': prob: 1.0000, logit: 30.62

Running Swap-In Intervention (swapping RAW MLP output at every requested layer from source to target at positions 'all'; edit_strength=1.0)...
Capturing source activations on prompt: 'Question: What is 58 + 83? Answer: 1' at positions [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]...
Running target pa

## Step 13 - Alternate No-Carry Control: 44 + 83

This tests whether the successful raw MLP swap is merely copying a source prompt whose tens digit is `3`. Here the no-carry source is `44 + 83 = 127`, so after the prefix `Answer: 1` the source next token should be `2`, not `3`. If patching this source into `58 + 83` pushes the target from `4` toward `2`, the effect is source-specific rather than a fixed `4 -> 3` artifact.


In [14]:
# Sanity check: token positions for the alternate no-carry source.
# 44 + 83 = 127, so after "Answer: 1" the next token should be 2.
!python src/intervention.py \
  --mode inhibit \
  --prompt "Question: What is 44 + 83? Answer: 1" \
  --target-token "4, 3, 2, 7" \
  --print-tokens


Loading model...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 184.31it/s]
Prompt token positions:
   0: id=14582    token='Question'
   1: id=25       token=':'
   2: id=3555     token=' What'
   3: id=374      token=' is'
   4: id=220      token=' '
   5: id=19       token='4'
   6: id=19       token='4'
   7: id=488      token=' +'
   8: id=220      token=' '
   9: id=23       token='8'
  10: id=18       token='3'
  11: id=30       token='?'
  12: id=21806    token=' Answer'
  13: id=25       token=':'
  14: id=220      token=' '
  15: id=16       token='1'
Use --positions last, --positions all, or comma-separated indices such as --positions 4,5,9


In [15]:
# SAE full latent control: 44+83 no-carry source -> 58+83 carry target.
# Desired direction: target 4 should fall and source-specific 2 should rise.
!python src/intervention.py \
  --mode swap \
  --source-prompt "Question: What is 44 + 83? Answer: 1" \
  --prompt "Question: What is 58 + 83? Answer: 1" \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_math_final_train_config.yaml \
  --target-token "4, 3, 2, 7" \
  --positions all \
  --edit-strength 2.0 \
  --output outputs/math_final_swap_full_44_83_to_58_83_strength2.json


Loading model...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 183.54it/s]

[1/3] Clean Model Baseline:
  - Top prediction: '4' (prob: 1.0000, logit: 30.62)
  - Target ' ': prob: 0.0000, logit: 14.31
  - Target '1': prob: 0.0001, logit: 21.75
  - Target '2': prob: 0.0003, logit: 22.38
  - Target '3': prob: 0.0008, logit: 23.50
  - Target '4': prob: 1.0000, logit: 30.62
  - Target '7': prob: 0.0000, logit: 19.50

Source baseline prediction on 'Question: What is 44 + 83? Answer: 1':
  - Target ' ': prob: 0.0000, logit: 14.62
  - Target '1': prob: 0.0003, logit: 20.50
  - Target '2': prob: 1.0000, logit: 28.75
  - Target '3': prob: 0.0008, logit: 21.62
  - Target '4': prob: 0.0001, logit: 19.38
  - Target '7': prob: 0.0000, logit: 18.12

Running Swap-In Intervention (swapping FULL latent code at every requested layer from source to target at positions 'all'; edit_strength=2

In [16]:
# Raw MLP control: 44+83 no-carry source -> 58+83 carry target.
# This is the cleanest test for source-specific patching: does the target move toward 2?
!python src/intervention.py \
  --mode swap \
  --source-prompt "Question: What is 44 + 83? Answer: 1" \
  --prompt "Question: What is 58 + 83? Answer: 1" \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_math_final_train_config.yaml \
  --target-token "4, 3, 2, 7" \
  --positions all \
  --raw-mlp-swap \
  --output outputs/math_final_swap_raw_mlp_44_83_to_58_83.json


Loading model...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 186.39it/s]

[1/3] Clean Model Baseline:
  - Top prediction: '4' (prob: 1.0000, logit: 30.62)
  - Target ' ': prob: 0.0000, logit: 14.31
  - Target '1': prob: 0.0001, logit: 21.75
  - Target '2': prob: 0.0003, logit: 22.38
  - Target '3': prob: 0.0008, logit: 23.50
  - Target '4': prob: 1.0000, logit: 30.62
  - Target '7': prob: 0.0000, logit: 19.50

Source baseline prediction on 'Question: What is 44 + 83? Answer: 1':
  - Target ' ': prob: 0.0000, logit: 14.62
  - Target '1': prob: 0.0003, logit: 20.50
  - Target '2': prob: 1.0000, logit: 28.75
  - Target '3': prob: 0.0008, logit: 21.62
  - Target '4': prob: 0.0001, logit: 19.38
  - Target '7': prob: 0.0000, logit: 18.12

Running Swap-In Intervention (swapping RAW MLP output at every requested layer from source to target at positions 'all'; edit_strength=1.0

In [17]:
# Raw MLP reverse control: 58+83 carry source -> 44+83 no-carry target.
# Desired direction: target 2 should fall and carry-source 4 should rise.
!python src/intervention.py \
  --mode swap \
  --source-prompt "Question: What is 58 + 83? Answer: 1" \
  --prompt "Question: What is 44 + 83? Answer: 1" \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_math_final_train_config.yaml \
  --target-token "4, 3, 2, 7" \
  --positions all \
  --raw-mlp-swap \
  --output outputs/math_final_swap_raw_mlp_58_83_to_44_83.json


Loading model...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 185.25it/s]

[1/3] Clean Model Baseline:
  - Top prediction: '2' (prob: 1.0000, logit: 28.75)
  - Target ' ': prob: 0.0000, logit: 14.62
  - Target '1': prob: 0.0003, logit: 20.50
  - Target '2': prob: 1.0000, logit: 28.75
  - Target '3': prob: 0.0008, logit: 21.62
  - Target '4': prob: 0.0001, logit: 19.38
  - Target '7': prob: 0.0000, logit: 18.12

Source baseline prediction on 'Question: What is 58 + 83? Answer: 1':
  - Target ' ': prob: 0.0000, logit: 14.31
  - Target '1': prob: 0.0001, logit: 21.75
  - Target '2': prob: 0.0003, logit: 22.38
  - Target '3': prob: 0.0008, logit: 23.50
  - Target '4': prob: 1.0000, logit: 30.62
  - Target '7': prob: 0.0000, logit: 19.50

Running Swap-In Intervention (swapping RAW MLP output at every requested layer from source to target at positions 'all'; edit_strength=1.0

## Step 14 - Persist final math outputs


In [18]:
# Copy final math graphs/results to Drive
import os, shutil, glob

drive_out = '/content/drive/MyDrive/mphil-project/outputs'
os.makedirs(drive_out, exist_ok=True)
for src in glob.glob('/content/test_run/outputs/math_final_*'):
    if os.path.isfile(src):
        shutil.copy2(src, drive_out)
        print('Copied', os.path.basename(src))
print('Done!')


Copied math_final_swap_full_44_83_to_58_83_strength2.json
Copied math_final_nocarry_54_83_3v4_graph.json
Copied math_final_swap_full_carry_to_nocarry.json
Copied math_final_carry_58_83_4v3_graph.html
Copied math_final_swap_raw_mlp_58_83_to_44_83.json
Copied math_final_nocarry_54_83_3v4_graph.md
Copied math_final_swap_full_nocarry_to_carry_strength2.json
Copied math_final_swap_sparse_carry_to_nocarry.json
Copied math_final_swap_full_nocarry_to_carry.json
Copied math_final_carry_58_83_4v3_graph.md
Copied math_final_swap_raw_mlp_carry_to_nocarry.json
Copied math_final_scan_carry_positive_allpos.json
Copied math_final_knockout_mlp_allpos_carry_58_83.json
Copied math_final_swap_sparse_nocarry_to_carry.json
Copied math_final_swap_raw_mlp_nocarry_to_carry.json
Copied math_final_nocarry_54_83_3v4_graph.html
Copied math_final_swap_raw_mlp_44_83_to_58_83.json
Copied math_final_carry_58_83_4v3_graph.json
Done!
